# 第5章 RCSJ モデルを解いて IV 曲線を描く

**この章でできるようになること**
- 研究の運動方程式 γ″ + βγ′ + sin γ = I + A(sin Ω₁τ + sin Ω₂τ) を RK4 で解ける
- 「はじめの部分を捨てて、後半を平均する」ことで電圧 V = ⟨γ′⟩ を求められる
- 電流の方向を NumPy でまとめて計算し（ベクトル化）、IV 曲線を描ける
- IV 曲線に「段（シャピロステップ）」が出ることを確かめられる

**例え：** 電圧を求めるのは「車の平均の速さ」を測るのに似ています。走り出し（**過渡応答**）は加速中なので測らずに捨て、落ち着いてからの区間で「進んだ距離 ÷ 時間」を測ります。研究コードの `N_run` が捨てる区間、`N_avg` が測る区間です。

| 用語 | 意味 | 例え |
|---|---|---|
| RCSJ モデル | 接合を「抵抗 R・容量 C・超伝導の電流」が並んだものとみなすモデル | 洗濯板の上を転がる、空気抵抗のある玉 |
| ジョセフソン接合 | 2つの超伝導体を薄い壁ではさんだ素子 | 2つの池をつなぐ細い水路 |
| IV 曲線 | 直流電流 I を変えたときの電圧 V のグラフ | 「アクセルの踏み方」と「速さ」の関係のグラフ |
| 過渡応答 | 計算を始めてすぐの、まだ落ち着いていない部分 | 車の走り出し |
| 定常状態 | 落ち着いて、同じような動きをくり返している状態 | 一定のペースで走っている車 |
| 時間平均 ⟨ ⟩ | ある区間の値を足して、点の数で割ったもの | 平均の速さ＝距離÷時間 |
| シャピロステップ | 交流を当てた接合の IV 曲線に出る、電圧が一定の段 | 坂道に刻まれた階段の踊り場 |
| 位相ロック | 接合の回転が、交流のリズムに引き込まれてそろうこと | 手拍子に合わせて歩く速さがそろう |

> 研究コードとのつながり：`py/rcsj.py` の `steady`（1つの電流）と、MATLAB の `matlab/60_発表/R8_oubutsu/pf_rk4_iv.m`（電流の方向をベクトル化）を、この章で順に作り直します。

In [ ]:
# 準備：教材の部品（kyozai/）を import できるようにする。どの章でも最初に1回だけ実行する。
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "kyozai").exists():   # kyozai/ があるフォルダまで上にたどる
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("教材の場所:", ROOT.name)

## 5.1 式と記号

$$\gamma'' + \beta\,\gamma' + \sin\gamma = I + A\,(\sin\Omega_1\tau + \sin\Omega_2\tau)$$

| 記号 | 意味 | 研究の標準値 |
|---|---|---|
| γ | 接合の位相差 | 初期値 0 |
| γ′ = G | 位相の変化の速さ。時間平均が電圧 V | 初期値 1.0 |
| β | 減衰 | 1〜12（この教材では主に 10） |
| I | 直流電流（臨界電流で規格化） | 掃引する |
| A | 2つの交流の振幅（2つとも同じ） | 0〜64 など |
| Ω₁, Ω₂ | 2つの交流の周波数 | 0.5, 0.8 |

（記号の出どころは研究リポジトリの `lab/式と前提.md` 1章）

**時間 τ の規格化について：** ′ は規格化した時間 τ での微分です。研究リポジトリの `lab/式と前提.md` では、この式の形は「ふつうプラズマ周波数 ω_p で時間を測った形と読むが、報告書には明記がない」とされていて、**未確認**です。この教材も研究コード（`py/rcsj.py`）と同じ式をそのまま使い、規格化についてはそれ以上のことを言いません。

## 5.2 1つの電流で解く

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

def accel(t, g, G, I, b, A, O1, O2):
    """γ″ を返す。"""
    return I - b * G - np.sin(g) + A * np.sin(O1 * t) + A * np.sin(O2 * t)

def run_one(I, b, A, O1=0.5, O2=0.8, dt=1/32, n_steps=4000, initial=1.0):
    """1つの電流 I について、γ′ の時系列を返す（教材用に短め）。"""
    g, G, t = 0.0, initial, 0.0
    Gs = np.empty(n_steps)
    for k in range(n_steps):
        k1 = accel(t, g, G, I, b, A, O1, O2);                         j1 = G
        k2 = accel(t + dt/2, g + dt/2*j1, G + dt/2*k1, I, b, A, O1, O2); j2 = G + dt/2*k1
        k3 = accel(t + dt/2, g + dt/2*j2, G + dt/2*k2, I, b, A, O1, O2); j3 = G + dt/2*k2
        k4 = accel(t + dt, g + dt*j3, G + dt*k3, I, b, A, O1, O2);       j4 = G + dt*k3
        g += dt/6*(j1 + 2*j2 + 2*j3 + j4)
        G += dt/6*(k1 + 2*k2 + 2*k3 + k4)
        t += dt
        Gs[k] = G
    return Gs

t0 = time.time()
Gs = run_one(I=3.0, b=10.0, A=0.0)
print(f"{time.time() - t0:.2f} 秒")
tau = np.arange(1, Gs.size + 1) / 32

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(tau, Gs)
ax.set_xlabel("τ"); ax.set_ylabel("γ' "); ax.set_title("β=10, I=3, A=0")
ax.grid(alpha=.3)
plt.show()

はじめは初期値 1.0 から急に変わり（過渡応答）、そのあと同じ形のゆれをくり返します（定常状態）。

## 5.3 捨てて、平均する

前半を捨てて後半を平均します。研究コードは N_run = N_avg = 20106 ステップ（dt = 1/32 なので τ にして約 628 ずつ）です。この節では短めにして、捨てる長さで結果がどう変わるかを見ます。

In [ ]:
Gs = run_one(I=3.0, b=10.0, A=0.0, n_steps=8000)
for N_run in [0, 100, 1000, 4000]:
    V = Gs[N_run:N_run + 4000].mean()
    print(f"N_run = {N_run:5d}  →  V = <γ'> = {V:.5f}")
print("参考：抵抗だけなら V ≈ I/β =", 3.0 / 10)

捨てないと初期値の影響が残り、少し違う値になります。十分に捨てれば値は落ち着きます。V が I/β より少し小さいのは、sin γ の項（超伝導の電流）が少し電流を受け持つからです。

## 5.4 for で IV 曲線を描く（遅い）

電流を変えて V を並べれば IV 曲線です。まずは素直に `for` で回します。ここでは時間を節約するため、点数と長さを減らしています。

In [ ]:
def voltage(I, b, A, n_run=1500, n_avg=1500, **kw):
    Gs = run_one(I, b, A, n_steps=n_run + n_avg, **kw)
    return max(Gs[n_run:].mean(), 0.0)    # 負なら 0（研究コードと同じ）

I_list = np.linspace(0, 4, 15)
t0 = time.time()
V_loop = np.array([voltage(I, 10.0, 0.0) for I in I_list])
t_loop = time.time() - t0
per_point = t_loop / I_list.size
print(f"for で {I_list.size} 点: {t_loop:.2f} 秒（1点あたり {per_point*1000:.0f} ミリ秒）")
# 研究と同じ長さ（20106+20106 ステップ。ここの約13倍）で、800点 × 33本を計算したら？
est = per_point * (40212 / 3000) * 800 * 33
print(f"研究なみの計算の見積もり: 約 {est/60:.0f} 分")

15 点なら一瞬ですが、研究と同じ長さで「800点 × 33本」を計算すると、上の見積もりのように長い時間がかかります。

## 5.5 電流の方向をまとめる（ベクトル化）

MATLAB の `pf_rk4_iv.m` と同じ考え方で、**全部の電流を配列にして同時に進めます**。時間の `for` は残りますが、電流の `for` は NumPy の配列計算に置きかわります。

In [ ]:
def iv_vectorized(Iarr, b, A, O1=0.5, O2=0.8, dt=1/32, n_run=1500, n_avg=1500, initial=1.0):
    I = np.asarray(Iarr, dtype=float)
    g = np.zeros_like(I)
    G = np.full_like(I, initial)
    t = 0.0
    S = np.zeros_like(I)
    f = lambda gg, GG, tt: I - b*GG - np.sin(gg) + A*np.sin(O1*tt) + A*np.sin(O2*tt)
    for k in range(n_run + n_avg):
        k1g = G;            k1G = f(g, G, t)
        k2g = G + dt/2*k1G; k2G = f(g + dt/2*k1g, G + dt/2*k1G, t + dt/2)
        k3g = G + dt/2*k2G; k3G = f(g + dt/2*k2g, G + dt/2*k2G, t + dt/2)
        k4g = G + dt*k3G;   k4G = f(g + dt*k3g, G + dt*k3G, t + dt)
        g = g + dt/6*(k1g + 2*k2g + 2*k3g + k4g)
        G = G + dt/6*(k1G + 2*k2G + 2*k3G + k4G)
        t += dt
        if k >= n_run:
            S += G
    return np.maximum(S / n_avg, 0.0)

t0 = time.time()
V_vec = iv_vectorized(I_list, 10.0, 0.0)
print(f"ベクトル化で {I_list.size} 点: {time.time() - t0:.2f} 秒")
print("for の結果と同じか:", np.allclose(V_loop, V_vec))

`lambda` は、その場で作る小さな名前なし関数です。`f = lambda x: x**2` は `def f(x): return x**2` と同じです。

15 点では差が小さいですが、点数を増やすと差が開きます。ベクトル化版は、点数を 20 倍にしても時間はあまり増えません（配列が長くなるだけ）。

In [ ]:
I_many = np.linspace(0, 4.5, 300)
t0 = time.time()
V0 = iv_vectorized(I_many, 10.0, 0.0)
print(f"ベクトル化で {I_many.size} 点: {time.time() - t0:.2f} 秒")

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(I_many, V0, label="A = 0")
ax.plot(I_many, I_many / 10, "k:", label="V = I/β (resistor only)")
ax.set_xlabel("I"); ax.set_ylabel("V = <γ'>"); ax.legend(); ax.grid(alpha=.3)
ax.set_title("β = 10, no drive")
plt.show()

I < 1 では V = 0（超伝導の電流だけで流れる）、I > 1 で電圧が出て、だんだん V = I/β に近づきます。

## 5.6 交流を当てるとステップが出る

A = 20 の二色の交流を当てます。研究の「β = 10、V = 0.3 の (−1, 1) ステップ」の周りを見るため、I = 1.6〜4.2 を計算します。

ここでは捨てる区間・平均する区間を**研究と同じ 20106 ステップ**にします。20106 × dt = 20106/32 ≈ 628.3 ≈ 200π で、これは第3章で見た二色の波形の共通周期 20π の**ほぼちょうど10回分**です（研究でこの数を選んだ理由は報告書に書かれていないので、ここでは「そうなっている」ことだけを言います）。平均する区間が周期のちょうど整数倍でないと、ゆれの「端数」が平均に残って、V が少しずれます（練習問題3）。

In [ ]:
I_win = np.linspace(1.6, 4.2, 300)
t0 = time.time()
V20 = iv_vectorized(I_win, 10.0, 20.0, n_run=20106, n_avg=20106)
print(f"{time.time() - t0:.1f} 秒")

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(I_win, V20, lw=1.5, label="A = 20")
ax.plot(I_win, iv_vectorized(I_win, 10.0, 0.0, n_run=20106, n_avg=20106), "gray", lw=1, label="A = 0")
for n, m in [(2, -1), (-1, 1), (-4, 3)]:     # V = 0.2, 0.3, 0.4
    Vs = n * 0.5 + m * 0.8
    if 0 < Vs < 0.45:
        ax.axhline(Vs, color="r", ls=":", lw=0.8)
        ax.text(1.62, Vs + 0.004, f"({n},{m}) V={Vs:.1f}", color="r", fontsize=8)
ax.set_xlabel("I"); ax.set_ylabel("V"); ax.legend(loc="lower right"); ax.grid(alpha=.3)
ax.set_title("β = 10, Ω1 = 0.5, Ω2 = 0.8")
plt.show()

交流を当てると、IV 曲線に**平らな段**ができます。段の高さは V = nΩ₁ + mΩ₂ にそろっています（赤い点線）。V = 0.3 の段が、差の周波数 Ω₂ − Ω₁ による **(−1, 1) ステップ** です。

**例え：** 階段の踊り場（段）では、電流（アクセル）を少し変えても電圧（速さ）が変わりません。接合の回転が交流のリズムに引き込まれて（位相ロック）、決まった速さでしか回れなくなっているからです。

## 第5章の MATLAB ↔ Python 対応

| やりたいこと | MATLAB | Python |
|---|---|---|
| 後半だけ平均 | `mean(G(N_run+1:end))` | `G[N_run:].mean()`（0 始まりなので `+1` が要らない） |
| 負なら 0 | `max(V, 0)` | `max(V, 0.0)`（数1つ）／`np.maximum(V, 0.0)`（配列） |
| 同じ形の 0 の配列 | `zeros(size(I))` | `np.zeros_like(I)` |
| 同じ形で値を入れる | `initial * ones(size(I))` | `np.full_like(I, initial)` |
| 名前のない関数 | `f = @(g, G, t) I - b*G - sin(g)` | `f = lambda g, G, t: I - b*G - np.sin(g)` |
| 時間を測る | `tic; ...; toc` | `t0 = time.time(); ...; time.time() - t0` |
| 電流をまとめて進める | `pf_rk4_iv.m`（`I` を配列にして RK4） | `iv_vectorized`（同じ考え方） |

全章の分をまとめたものは [`対応表.md`](../対応表.md) にあります。

## 穴埋め練習

5.3 の「捨てて、平均する」を自分で書きます。A = 0 のときは V ≈ I/β に近い値になるはずです。

In [ ]:
# 穴埋め：___ を埋めてから、全部の行を選んで Cmd + /（Windows は Ctrl + /）を押し、行頭の # を外して実行する
# Gs = run_one(I=2.0, b=10.0, A=0.0, n_steps=6000)
# N_run = 2000
# V = Gs[___:].mean()
# print(f"V = {V:.4f}   参考 I/β = {2.0 / ___:.4f}")

<details><summary>穴埋めの答え（クリックで開く）</summary>

1つ目は `N_run`（はじめの 2000 ステップを捨てる）、2つ目は β の `10.0` です。V は I/β = 0.2 より少し小さくなります（5.3 の説明と同じ理由）。
</details>

下のセルは答えを埋めた版です。自分で埋めた結果と同じになるか確かめてください。

In [ ]:
# 答え合わせ（埋めた版）
Gs = run_one(I=2.0, b=10.0, A=0.0, n_steps=6000)
N_run = 2000
V = Gs[N_run:].mean()
print(f"V = {V:.4f}   参考 I/β = {2.0 / 10.0:.4f}")

## 予想してから実行

**問題：** 交流なし（A = 0）、β = 10 で、I = 0.5 と I = 4.0 のときの V はいくつくらいでしょうか。5.5 の IV 曲線を思い出して予想してください。

**先に予想を紙かメモに書いてから**、次のセルを実行してください。

In [ ]:
for I_ in [0.5, 4.0]:
    print(f"I = {I_}: V = {voltage(I_, 10.0, 0.0):.4f}   (I/β = {I_ / 10:.2f})")

<details><summary>解説（実行してから開く）</summary>

- I = 0.5 では **V はほぼ 0**。I < 1 では超伝導の電流（sin γ の項）だけで電流が流れ、位相は止まります（4.8 の「くぼみで止まる玉」）。出力が 0.0001 のようにぴったり 0 にならないのは、この節の短い計算（捨てる区間 1500 ステップ）では、初期値 γ′ = 1 の影響が少し残るためと考えられます（推測。確かめていない）。
- I = 4.0 では **0.4 より少し小さい値**。I が大きいほど V = I/β に近づきますが、sin γ の項が少し電流を受け持つ分だけ小さくなります。
</details>

## 練習問題

1. 5.6 で A = 0, 10, 20, 30 の IV 曲線を重ねて描いてください。V = 0.3 の段の長さは A で変わりますか。（1本 数秒かかります）
2. `iv_vectorized` の `initial`（γ′ の初期値）を 0 にすると、A = 0 の IV 曲線は変わりますか。β = 10 は減衰が大きい（過減衰）ことに注意して考えてください。
3. 5.6 の A = 20 を、`n_run=3000, n_avg=3000`（共通周期の整数倍でない長さ）で計算し直して、段の高さが 0.3 からずれることを確かめてください。

<details><summary>答え</summary>

```python
# 1.
fig, ax = plt.subplots(figsize=(6, 4.5))
for A in [0, 10, 20, 30]:
    ax.plot(I_win, iv_vectorized(I_win, 10.0, A, n_run=20106, n_avg=20106), label=f"A={A}")
ax.axhline(0.3, color="r", ls=":"); ax.legend(); plt.show()
```
段の長さは A で変わります（A=0 では段がない）。A による変わり方は第7・8章で詳しく見ます。

2. ほとんど変わりません。過減衰では、どこから始めても同じ定常状態に早く落ち着くので、初期値の影響は捨てる区間で消えます（減衰が小さいと、ヒステリシスで初期値によって結果が変わることがあります）。

3. 段の上でも γ′ は交流に合わせてゆれているので、平均する区間に周期の「端数」があると、その分だけ平均がずれます。A が大きいほどゆれも大きいので、ずれも大きくなります。
</details>

## まとめ
- 電圧 V は、過渡応答を捨てたあとの γ′ の時間平均。研究では N_run = N_avg = 20106 ステップ（≈ 200π、二色の共通周期 20π の10回分）。
- 電流の方向を配列にまとめると、同じ RK4 のまま速くなる（MATLAB の `pf_rk4_iv.m` と同じ考え方）。
- 交流を当てると、V = nΩ₁ + mΩ₂ に平らな段（シャピロステップ）ができる。

それでも研究と同じ長さ（20106 + 20106 ステップ）で何千点も計算するには、まだ遅すぎます。次の章では **numba** でさらに速くします。